In [ ]:
from langgraph.graph import START, END, StateGraph
# from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import SystemMessage, HumanMessage
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()

In [ ]:
class EvaluateFormate(BaseModel):
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str = Field(description="Feedback for the tweet.")

In [ ]:
generator_llm = ChatGoogleGenerativeAI(model="gemini-flash-lite-latest")
evaluator_llm = ChatGoogleGenerativeAI(model="gemini-flash-latest")
structured_evaluator_llm = evaluator_llm.with_structured_output(EvaluateFormate)
optimizer_llm = ChatGoogleGenerativeAI(model="gemini-flash-lite-latest")

In [ ]:
class TweerState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

In [ ]:
def generate_tweet(state: TweerState):
    # prompt
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""Write a short, original, and hilarious tweet on topic "{state['topic']}:".
         
Rules:
- Do NOT use question-answer formate.
- Max 250 characters.
- Use observational humor, irony, sarcasm, or culture references.
- Think in meme login, punchlines, or relatable tasks.
- Use simple, day to day english""")
    ]

    tweet = generator_llm.invoke(messages).text

    return {'tweet': tweet}







def evaluate_tweet(state: TweerState):
    messages = [
        SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
        HumanMessage(content=f"""Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality - Is this fresh, or have you seen it a hundred times before?
2. Humor - Did it genuinely make you smile, laugh, or chuckle?
3. Punchiness - Is it short, sharp, and scroll-stopping?
4. Virality Potential - Would people retweet or share it?
5. Format - Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 260 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"
- feedback: One paragraph explaining the strengths and weaknesses
""")

    ]

    response = structured_evaluator_llm.invoke(messages)

    return {"evaluation": response.evaluation, "feedback": response.feedback}






def optimize_tweet(state: TweerState):
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 250 characters.
""")
    ]

    improved_tweet = optimizer_llm.invoke(messages).text
    iteration = state["iteration"] + 1

    return {"tweet": improved_tweet, "iteration": iteration}





def check_evaluation(state: TweerState):
    if state["evaluation"] == "approved" or state["iteration"] >= state["max_iteration"]:
        return "approved"
    else:
        return "need_improvement"

In [ ]:
graph = StateGraph(TweerState)


graph.add_node("Generate", generate_tweet)
graph.add_node("Evaluate", evaluate_tweet)
graph.add_node("Optimize", optimize_tweet)


graph.add_edge(START, "Generate")
graph.add_edge("Generate", "Evaluate")
graph.add_conditional_edges("Evaluate", check_evaluation, {"approved": END, "need_improvement": "Optimize"})       # path_map  (source, path, path_map)
graph.add_edge("Optimize", "Evaluate")


workflow = graph.compile()


In [ ]:
initial_state = {
    "topic": "Water Bottle", 
    "iteration": 1,
    "max_iteration": 5
}

final_state = workflow.invoke(initial_state)

In [ ]:
print("iteration:", final_state["iteration"])
print("evaluation:", final_state["evaluation"])
print("tweet:", final_state["tweet"])
print("\n\nfeedback:", final_state["feedback"])